In [ ]:
# Put import statements here
import os
# hide tensorflow info/warning logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import sys
import subprocess
from pathlib import Path
import json


# Local files/code
import src.data_preprocessing.image_preprocessing as img_pre
import src.data_preprocessing.text_preprocessing as text_pre
import src.data_preprocessing.text_data_exploration as text_explore
from src.agents.Visual_Agent import VisualModel
from src.util.logger import Logger
from src.util.config import config, PROJECT_ROOT
import src.util.general as general_util
from src.data_preprocessing.TextTokenizer import TextTokenizer
from src.data_preprocessing.TextEmbedder import TextEmbedder
import src.evaluation.visual_evaluator as visual_evaluator


In [ ]:
# Determine the project's data repo location
DATA_DIR= PROJECT_ROOT / "data"


In [ ]:
# example of how to chunk+vectorize text data

USDA_data_path= DATA_DIR / "EOI_Data_Pulls"
USDA_plant_sheets_path= USDA_data_path / "plant_sheets"
temp= USDA_plant_sheets_path / "ABBA/ABBA_1.pdf"

# load and clean the text data however you need to
text= text_pre.load_pdf_text_data(temp)
text= text_pre.replace_urls(text, "url")
text= text.replace("<", "")
text= text.replace(">", "")

# make the text embedder
embedder= TextEmbedder("sentence-transformers/all-MiniLM-L6-v2")

CHUNK_METHOD= "word" # word or sentence
CHUNK_MAX_SIZE= 64
CHUNK_OVERLAP_AMOUNT= 16

# chunk and encode
records= embedder.chunk_and_encode(text, CHUNK_METHOD, CHUNK_MAX_SIZE, CHUNK_OVERLAP_AMOUNT)
# This takes ~30sec - 1.5min to run
# parse the USDA plant sheets, chunk them, vectorize them, and make organized records
# Also chunk/vectorize USDA json files that have relevant plant sheets
# NOTE: the PDF parsing library loves to spit out annoying logs
#       I've tried to supress them a million ways, but could not get it to stop
#       The logs are annoying but superficial
records= process_USDA_plant_sheets(
    USDA_data_path,
    embedder,
    CHUNK_METHOD,
    CHUNK_MAX_SIZE,
    CHUNK_OVERLAP_AMOUNT
)

# review records
for record in records:
    Logger.info(f"\n{record}")



In [ ]:
# Example of one of the records for one chunk of a plant
Logger.info(f"records type: {type(records)}")
Logger.info(f"records len: {len(records)}")
Logger.info(f"Example of a pdf chunk record: \n\n{records['ABBA'][0]}")
Logger.info(f"Example of a json record: \n\n{records['ABBA'][-1]}")

In [ ]:
# load the plantExpertVQA training, validation, and testing datasets
plant_expert_vqa_data_path= DATA_DIR / config.data.PlantExpertVQA.data_path
plant_expert_vqa_path= DATA_DIR / "PlantExpertVQA"
train_data_path= plant_expert_vqa_data_path / config.data.PlantExpertVQA.train_file
test_data_path= plant_expert_vqa_data_path / config.data.PlantExpertVQA.test_file
validation_data_path= plant_expert_vqa_data_path / config.data.PlantExpertVQA.validation_file


plant_expert_vqa_TRAIN= text_pre.load_csv(train_data_path)
plant_expert_vqa_TEST= text_pre.load_csv(test_data_path)
plant_expert_vqa_VAL= text_pre.load_csv(validation_data_path)

In [ ]:

# Get the configured preprocessing parameters
PEVQA_text_columns= general_util.parse_list_from_string(config.data.PlantExpertVQA.text_columns)
PEVQA_columns_to_remove= general_util.parse_list_from_string(config.data.PlantExpertVQA.columns_to_remove)
PEVQA_na_fill= vars(config.data.PlantExpertVQA.na_fill)

# preprocess (not tokenize) the training, testing, and validation datasets
# NOTE: the stop word removal may be too intense.  We most likely want to fine tune the stopword set, or define our own set
#       Right now it removes words like "what" and "why", which will most likely be bad for a VQA system that answers questions.

# training
text_pre.preprocess_dataframe(
    plant_expert_vqa_TRAIN,
    PEVQA_text_columns,
    PEVQA_columns_to_remove,
    PEVQA_na_fill,
    ["image_path"],
    plant_expert_vqa_path,
)

# testing
text_pre.preprocess_dataframe(
    plant_expert_vqa_TEST,
    PEVQA_text_columns,
    PEVQA_columns_to_remove,
    PEVQA_na_fill,
    ["image_path"],
    plant_expert_vqa_path,
)

#validation
text_pre.preprocess_dataframe(
    plant_expert_vqa_VAL,
    PEVQA_text_columns,
    PEVQA_columns_to_remove,
    PEVQA_na_fill,
    ["image_path"],
    plant_expert_vqa_path,
)


In [ ]:

column_distribution_args= [
    {"column": "crop", "show_counts": False, "figure_size": (10, 5)},
    {"column": "severity", "show_counts": True, "figure_size": (5, 5)},
    {"column": "category", "show_counts": True, "figure_size": (5, 5)},
    {"column": "answer_type", "show_counts": True, "figure_size": (5, 5)},
    {"column": "question_category", "show_counts": False, "figure_size": (10, 5)},
]

# explore the cleaned training dataset
text_explore.explore_data(
    plant_expert_vqa_TRAIN, 
    column_distribution_args, 
    PEVQA_text_columns, 
    top_n_words=20, 
    name="Plant Expert VQA Training Dataset"
)

In [ ]:
# Testing the tokenizer and showing how to use it

# intialize the tokenizer
tokenizer= TextTokenizer("distilbert-base-uncased", 128)

# example of the passing in a single string to tokenize
inputs, attention_mask= tokenizer.encode_text("hello")
Logger.info(inputs)

# how to decode a single tensor output
output= tokenizer.decode(inputs)
Logger.info(output)

# encoding a list of strings
inputs, attention_mask= tokenizer.encode_text(["hello", "this is a test", "i am putting in multiple strings"])
Logger.info(inputs)

# decoding a list of tensors
output= tokenizer.batch_decode(inputs)
Logger.info(output)

# encoding a column of data
questions= plant_expert_vqa_TRAIN["question_text"]
inputs, attention_mask= tokenizer.encode_text(questions)

# decoding a all of those tensors
output= tokenizer.batch_decode(inputs)
Logger.info(f"first few decoded Tensors: \n\n{output[:10]}")


In [ ]:
# Adding on macbook - will need to retest on workstation later tonight due to GPU acceleration
IMAGES_ROOT = DATA_DIR / "PlantExpertVQA"
traits = ["crop", "disease", "category", "severity"]

classes = {}
for t in traits:
    names = sorted(plant_expert_vqa_TRAIN[t].unique())  # adds names of plants
    classes[t] = names + ["unknown"]  # adds unknown species

train_img = VisualModel.one_row_per_image(plant_expert_vqa_TRAIN, traits)
val_img = VisualModel.one_row_per_image(plant_expert_vqa_VAL, traits)
test_img = VisualModel.one_row_per_image(plant_expert_vqa_TEST, traits)
classes = VisualModel.build_classes(train_img, traits)

train_ds = VisualModel.make_dataset(train_img, classes, IMAGES_ROOT, training=True)
val_ds = VisualModel.make_dataset(val_img, classes, IMAGES_ROOT)

visual_model = VisualModel(classes=classes)
visual_model.build()
visual_model.compile()
history_frozen = visual_model.fit(train_ds, val_ds, epochs=20)
visual_model.unfreeze()
history_tuned = visual_model.fit(train_ds, val_ds, epochs=5)

visual_evaluator.plot_loss(history_frozen, history_tuned)
visual_evaluator.plot_accuracy(history_frozen, history_tuned)
visual_evaluator.plot_accuracy(history_frozen, history_tuned, heads=["crop", "disease"])

In [ ]:
import json
Path("./models/visual_models/9_16_classes.json").write_text(json.dumps(classes))
visual_model.save(path="./models/visual_models/9_16.keras")

In [ ]:
MODEL_DIR = Path("./models/visual_models")
IMAGES_ROOT = DATA_DIR / "PlantExpertVQA"

vm = VisualModel.load(MODEL_DIR / "9_16.keras", MODEL_DIR/"9_16_classes.json")

In [ ]:
test_table, test_predictions = visual_evaluator.evaluate(vm, plant_expert_vqa_TEST, IMAGES_ROOT, name="Test_VisualMoel")
test_table